# Financial Risk Analysis: VaR Modeling and Backtesting

## Objective
This project aims to estimate and evaluate market risk using Value-at-Risk (VaR) models.

We implement multiple approaches including:
- Historical VaR
- Parametric VaR
- EWMA-based VaR
- GARCH-based VaR

Model performance is evaluated using statistical backtesting methods.

## 1. Data Preparation

We construct a portfolio consisting of three major technology stocks:
- Apple (AAPL)
- Microsoft (MSFT)
- Google (GOOGL)

Portfolio returns are computed using log returns and fixed weights.

In [ ]:
import numpy as np

from src.data_loader import (
    load_prices,
    compute_log_returns,
    build_portfolio_returns
)

tickers = ["AAPL", "MSFT", "GOOGL"]
weights = np.array([0.5, 0.3, 0.2])

prices = load_prices(
    tickers=tickers,
    start="2018-01-01",
    end="2023-12-31"
)

returns = compute_log_returns(prices)
portfolio_returns = build_portfolio_returns(returns, weights)

ModuleNotFoundError: No module named 'numpy'

## 2. Rolling VaR Estimation and Backtesting

We estimate VaR using a rolling window approach (250 days) to capture time-varying risk.

Two models are implemented:
- Historical simulation
- Parametric (normal distribution)

Backtesting is conducted using:
- Kupiec test (unconditional coverage)
- Christoffersen test (independence of exceptions)

In [ ]:
from src.rolling_var import rolling_var_estimation
from src.backtesting import (
    compute_exceptions,
    exception_rate,
    kupiec_test,
    christoffersen_test,
    plot_rolling_var
)

In [ ]:
methods = ["historical", "parametric"]

results = {}

for method in methods:
    rolling_var = rolling_var_estimation(
        portfolio_returns,
        window=250,
        alpha=0.99,
        method=method
    )

    aligned_returns = portfolio_returns.loc[rolling_var.index]

    exceptions = compute_exceptions(aligned_returns, rolling_var)

    p_hat, x, t = exception_rate(exceptions)

    lr_uc, p_uc = kupiec_test(x, t, alpha=0.01)
    lr_cc, p_cc = christoffersen_test(exceptions)

    results[method] = {
        "exceptions": x,
        "exception_rate": p_hat,
        "kupiec_p": p_uc,
        "christoffersen_p": p_cc
    }

    print(method, results[method])

    plot_rolling_var(
        aligned_returns,
        rolling_var,
        exceptions,
        title=f"Rolling VaR Backtest ({method})"
    )

### Interpretation

The exception rate should be close to the theoretical level (1% for 99% VaR).

- A high p-value in the Kupiec test indicates correct coverage
- A high p-value in the Christoffersen test indicates independence of violations

Together, these tests evaluate whether the VaR model is statistically reliable.

## 3. EWMA-based VaR

We use the Exponentially Weighted Moving Average (EWMA) model to estimate volatility.

This approach assigns higher weights to recent observations, capturing volatility clustering.

In [ ]:
from src.volatility import ewma_volatility, ewma_var

ewma_vol = ewma_volatility(
    returns=portfolio_returns,
    lambda_=0.94,
    window=250
)

ewma_var_99 = ewma_var(
    ewma_vol=ewma_vol,
    alpha=0.99
)

In [ ]:
aligned_returns = portfolio_returns.loc[ewma_var_99.index]

exceptions = compute_exceptions(aligned_returns, ewma_var_99)
p_hat, x, t = exception_rate(exceptions)

lr_uc, p_uc = kupiec_test(x, t, alpha=0.01)
lr_ind, p_ind = christoffersen_test(exceptions)

plot_rolling_var(
    aligned_returns,
    ewma_var_99,
    exceptions,
    title="EWMA VaR (99%) Backtesting"
)

## 4. GARCH-based VaR

We implement a rolling GARCH(1,1) model to estimate conditional volatility.

This model captures:
- Volatility clustering
- Time-varying variance

Compared to EWMA, GARCH provides a more flexible framework for modeling risk dynamics.

In [ ]:
from src.volatility import rolling_garch_volatility, garch_var

garch_vol = rolling_garch_volatility(
    returns=portfolio_returns,
    window=250,
    refit_every=5,
    p=1,
    q=1,
    dist="normal"
)

garch_var_99 = garch_var(
    garch_vol=garch_vol,
    alpha=0.99
)

In [ ]:
aligned_returns = portfolio_returns.loc[garch_var_99.index]

exceptions = compute_exceptions(aligned_returns, garch_var_99)

p_hat, x, t = exception_rate(exceptions)

lr_uc, p_uc = kupiec_test(x, t, alpha=0.01)
lr_ind, p_ind = christoffersen_test(exceptions)

plot_rolling_var(
    aligned_returns,
    garch_var_99,
    exceptions,
    title="Rolling GARCH(1,1) VaR Backtesting"
)

## 5. Stress Testing and Scenario Analysis

In addition to VaR, we perform stress testing to evaluate extreme risk scenarios.

This includes:
- Historical worst-case losses
- Hypothetical market shocks
- Volatility shock scenarios

Stress testing complements VaR by capturing tail risks beyond normal assumptions.

In [ ]:
from src.stress_testing import stress_testing_module

stress_table = stress_testing_module(
    returns=portfolio_returns,
    portfolio_value=1_000_000,
    alpha_levels=(0.05, 0.01),
    worst_n_days=10,
    worst_percentile=0.01,
    hypothetical_shocks=(-0.10, -0.20, -0.30),
    vol_shock_multiplier=2.0
)

## Key Insight

A critical implementation detail in VaR backtesting is the sign convention.

VaR represents a left-tail loss (negative value), so an exception occurs when:

return < VaR

Incorrectly using return < -VaR can lead to an inflated exception rate (~99%), invalidating statistical tests.

This highlights the importance of correctly interpreting risk metrics in practice.